# Predicción de supervivencia del Titanic con Machine Learning

## Unidad 05

### Machine Learning con Scikit-learn

Dataset: Titanic - Machine Learning from Disaster  
Fuente: Kaggle  
Modelo: Random Forest Classifier

**Estudiante:** Al Farias Leyva  
**Matrícula:** 230389  
**Grupo:** 9A IDGS  
**Institución:** Universidad Tecnológica de Xicotepec de Juárez

## 1. Objetivo

El propósito es construir un modelo de aprendizaje supervisado que clasifique a los pasajeros en dos grupos: sobrevivió o no sobrevivió. Se utilizará el conjunto de entrenamiento oficial de Kaggle y un pipeline de Scikit-learn para mantener unidos el preprocesamiento y el modelo.

## 2. Importaciones

Pandas se utiliza para manipular los datos. Las herramientas de Scikit-learn permiten dividir el dataset, transformar columnas, entrenar el Random Forest y evaluar sus predicciones.

In [ ]:
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

## 3. Carga y validación del dataset

El notebook busca el archivo tanto desde la raíz del repositorio como desde la carpeta del notebook. Si no existe, se detiene con instrucciones claras: no genera ni reemplaza datos. También verifica que estén presentes las columnas requeridas.

In [ ]:
candidate_paths = [
    Path("Practica10/data/train.csv"),
    Path("../data/train.csv"),
]

data_path = next(
    (path for path in candidate_paths if path.exists()),
    candidate_paths[0],
)

if not data_path.exists():
    raise FileNotFoundError(
        "No se encontró el dataset oficial. Descarga train.csv desde "
        "https://www.kaggle.com/competitions/titanic/data y colócalo en "
        "Practica10/data/train.csv."
    )

df = pd.read_csv(data_path)

required_columns = {
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "Survived",
}
missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise ValueError(
        "El CSV no contiene las columnas requeridas: "
        + ", ".join(sorted(missing_columns))
    )

print(f"Dataset cargado desde: {data_path.resolve()}")
print(f"Registros cargados: {len(df)}")

## 4. Exploración de datos

Antes de entrenar se revisan ejemplos, dimensiones, tipos de datos, valores faltantes y balance de clases. Esta inspección ayuda a decidir qué transformaciones necesita cada variable.

In [ ]:
df.head()

Las primeras filas permiten comprobar visualmente que cada registro corresponde a un pasajero y que las columnas esperadas fueron leídas correctamente.

In [ ]:
df.shape

La tupla anterior muestra, en ese orden, la cantidad de filas y columnas disponibles en el archivo.

In [ ]:
df.info()

El resumen permite distinguir columnas numéricas y categóricas, además de comparar la cantidad de valores no nulos. Las diferencias entre el total de filas y los valores no nulos señalan datos faltantes.

In [ ]:
df.isnull().sum()

Este conteo identifica exactamente qué variables tienen valores faltantes. El pipeline los tratará sin eliminar pasajeros: mediana para variables numéricas y categoría más frecuente para variables categóricas.

In [ ]:
survival_counts = df["Survived"].value_counts().sort_index()
survival_percentages = (
    df["Survived"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

pd.DataFrame({
    "Cantidad": survival_counts,
    "Porcentaje": survival_percentages,
}).rename(index={0: "No sobrevivió", 1: "Sobrevivió"})

La tabla muestra la cantidad y proporción de cada clase. Esta revisión justifica usar una división estratificada, de forma que entrenamiento y prueba conserven aproximadamente la misma distribución.

In [ ]:
df.groupby("Sex")["Survived"].agg(
    Pasajeros="count",
    Tasa_supervivencia="mean",
).assign(
    Tasa_supervivencia_porcentaje=lambda table: (
        table["Tasa_supervivencia"] * 100
    ).round(2)
)

La tasa es la media de una variable codificada con 0 y 1, por lo que también representa la proporción de supervivencia de cada categoría de sexo. La salida permite comparar los grupos sin confundir cantidad de pasajeros con porcentaje.

In [ ]:
df.groupby("Pclass")["Survived"].agg(
    Pasajeros="count",
    Tasa_supervivencia="mean",
).assign(
    Tasa_supervivencia_porcentaje=lambda table: (
        table["Tasa_supervivencia"] * 100
    ).round(2)
)

La tabla compara la supervivencia entre primera, segunda y tercera clase. Esta diferencia exploratoria muestra por qué Pclass puede aportar información al clasificador, aunque por sí sola no determina el resultado de una persona.

## 5. Variables del modelo

Se usan siete características disponibles antes de conocer el resultado. Survived es la variable objetivo: 0 significa que no sobrevivió y 1 que sobrevivió.

In [ ]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
]
target = "Survived"

X = df[features]
y = df[target]

print("Características:", features)
print("Variable objetivo:", target)
print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

## 6. División de entrenamiento y prueba

El 80 % de los registros se utiliza para que el modelo aprenda y el 20 % restante para medir su desempeño con ejemplos que no vio durante el entrenamiento. stratify conserva la proporción de las dos clases y random_state hace reproducible la división.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Registros de entrenamiento: {len(X_train)}")
print(f"Registros de prueba: {len(X_test)}")
print(
    "Proporción de supervivencia en entrenamiento: "
    f"{y_train.mean():.2%}"
)
print(
    "Proporción de supervivencia en prueba: "
    f"{y_test.mean():.2%}"
)

## 7. Preprocesamiento

Las variables numéricas reciben una imputación por mediana. En las categóricas se completa el valor más frecuente y luego se crean columnas binarias con OneHotEncoder. handle_unknown="ignore" permite predecir aunque aparezca una categoría no vista durante el entrenamiento.

In [ ]:
numeric_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
]
categorical_features = [
    "Sex",
    "Embarked",
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features),
])

preprocessor

## 8. Modelo y pipeline

RandomForestClassifier combina numerosos árboles de decisión. Se emplean 200 árboles y una semilla fija para obtener un resultado reproducible. El pipeline garantiza que el mismo preprocesamiento se aplique durante el entrenamiento, la evaluación y las pruebas manuales.

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", model),
])

pipeline

## 9. Entrenamiento

fit calcula las imputaciones y categorías únicamente con el conjunto de entrenamiento; después transforma esos datos y ajusta los 200 árboles del clasificador.

In [ ]:
pipeline.fit(X_train, y_train)

print("Entrenamiento completado correctamente.")

## 10. Predicciones

El modelo procesa el conjunto de prueba y produce una clase para cada pasajero. La siguiente tabla compara diez valores reales con su predicción.

In [ ]:
y_pred = pipeline.predict(X_test)

comparison = pd.DataFrame({
    "Real": y_test.iloc[:10].map({
        0: "No sobrevivió",
        1: "Sobrevivió",
    }).to_numpy(),
    "Predicción": pd.Series(y_pred[:10]).map({
        0: "No sobrevivió",
        1: "Sobrevivió",
    }),
})

comparison

## 11. Evaluación

### Accuracy

Accuracy es la proporción total de predicciones correctas. Se presenta como decimal y como porcentaje con dos decimales.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy:.2%}")

### Matriz de confusión

Las filas representan los valores reales y las columnas las predicciones. La diagonal contiene los aciertos; las otras celdas muestran los dos tipos de error.

In [ ]:
confusion = confusion_matrix(y_test, y_pred)

confusion_df = pd.DataFrame(
    confusion,
    index=[
        "Real: No sobrevivió",
        "Real: Sobrevivió",
    ],
    columns=[
        "Predicción: No sobrevivió",
        "Predicción: Sobrevivió",
    ],
)

confusion_df

### Classification report

Precision indica qué proporción de las predicciones de una clase fue correcta. Recall indica qué proporción de los casos reales de esa clase fue encontrada. F1-score equilibra precision y recall. Accuracy resume el porcentaje correcto considerando ambas clases.

In [ ]:
report = classification_report(
    y_test,
    y_pred,
    target_names=[
        "No sobrevivió",
        "Sobrevivió",
    ],
    digits=4,
)

print(report)

## 12. Prueba con pasajeros nuevos

Se crean dos perfiles ficticios con características diferentes. No es necesario transformar sus columnas manualmente porque el pipeline aplica todas las operaciones antes de predecir.

In [ ]:
new_passengers = pd.DataFrame([
    {
        "Pclass": 3,
        "Sex": "male",
        "Age": 22,
        "SibSp": 1,
        "Parch": 0,
        "Fare": 7.25,
        "Embarked": "S",
    },
    {
        "Pclass": 1,
        "Sex": "female",
        "Age": 38,
        "SibSp": 1,
        "Parch": 0,
        "Fare": 71.28,
        "Embarked": "C",
    },
])

manual_predictions = pipeline.predict(new_passengers)
prediction_labels = {
    0: "No sobrevivió",
    1: "Sobrevivió",
}

manual_results = new_passengers.copy()
manual_results.insert(
    0,
    "Pasajero",
    range(1, len(manual_results) + 1),
)
manual_results["Predicción"] = [
    prediction_labels[value] for value in manual_predictions
]

manual_results

Estas respuestas son predicciones estadísticas construidas a partir de patrones del conjunto de entrenamiento. No representan una certeza histórica ni deben interpretarse como una regla sobre personas reales.

## 13. Conclusión

Este ejercicio permite seguir un proceso completo de clasificación supervisada: revisar datos, separar variables, evitar fugas de información, preprocesar valores faltantes y categorías, entrenar un modelo y evaluarlo con varias métricas. El pipeline mantiene el trabajo reproducible y facilita aplicar el modelo a nuevos registros. Al interpretar el resultado es importante considerar tanto los aciertos como los errores y recordar que el modelo aprende asociaciones presentes en un dataset histórico.